<a href="https://colab.research.google.com/github/nsasto/echo/blob/main/goEcho.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# goEcho

you'll need a kaggle account to access the fine tuned model.

1. Go to Kaggle: Log in to your Kaggle account (or create one if you don't have one).

2. Navigate to your profile: Click on your profile picture in the top-right corner, then select "Account."

3. Create New API Token: Scroll down to the "API" section.

If you have an existing token, you might see an "Expire API Token" button. Click it first to revoke the old one.

Then, click the "Create New API Token" button.

Download kaggle.json: This will download a file named kaggle.json to your computer. This file contains your username and API key. Keep this file secure, as it grants access to your Kaggle account.

Upload our kaggle.json API file


In [ ]:
from google.colab import files

files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"nathansasto","key":"fc6550be85a309f54202b321a3ac2ee7"}'}

In [ ]:
# Create a directory for Kaggle configuration if it doesn't exist
!mkdir -p ~/.kaggle/

# Copy the uploaded kaggle.json file to the Kaggle configuration directory
!cp kaggle.json ~/.kaggle/

# Set appropriate permissions for the kaggle.json file
# This is crucial for security and Kaggle API functionality (read/write by owner only)
!chmod 600 ~/.kaggle/kaggle.json

# Install the Kaggle API client (if not already installed)
!pip install -q kaggle

Download the dataset (our custom whisper model)

In [ ]:
# Download the dataset. The -d flag specifies a dataset.
# The dataset will be downloaded as a zip file to the current working directory (/content/ by default).
!kaggle datasets download -d nathansasto/whisper-echo

Dataset URL: https://www.kaggle.com/datasets/nathansasto/whisper-echo
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
 97% 824M/854M [00:08<00:00, 296MB/s]
100% 854M/854M [00:08<00:00, 103MB/s]


In [ ]:
# Create a directory for the unzipped contents
!mkdir -p whisper_echo

# Unzip the downloaded file into the new directory
!unzip whisper-echo.zip -d whisper_echo

Archive:  whisper-echo.zip
  inflating: whisper_echo/added_tokens.json  
  inflating: whisper_echo/config.json  
  inflating: whisper_echo/generation_config.json  
  inflating: whisper_echo/merges.txt  
  inflating: whisper_echo/model.safetensors  
  inflating: whisper_echo/normalizer.json  
  inflating: whisper_echo/preprocessor_config.json  
  inflating: whisper_echo/special_tokens_map.json  
  inflating: whisper_echo/tokenizer_config.json  
  inflating: whisper_echo/training_args.bin  
  inflating: whisper_echo/vocab.json  


In [ ]:
import os
import torch
import pandas as pd
from datasets import Dataset, Audio
from transformers import (
    pipeline,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import torch

Install dependency requirements for pyaudio in colab

In [ ]:
%%capture
#in colab
!apt-get install -y portaudio19-dev
!pip install pyaudio

In [ ]:
import pyaudio
import wave
import librosa
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor, pipeline

In [ ]:

# Audio recording settings
CHUNK = 1024
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 16000
WAVE_OUTPUT_FILENAME = "temp_recording.wav"
MODEL_PATH = "/content/whisper_echo"

In [ ]:

# Load Whisper model and processor
print("🔹 Loading fine-tuned model...")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH)
processor = WhisperProcessor.from_pretrained(MODEL_PATH)
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0 if torch.cuda.is_available() else -1,
    generate_kwargs={"forced_decoder_ids": None}
)
print("✅ Fine-tuned model loaded successfully")


🔹 Loading fine-tuned model...


Device set to use cpu


✅ Fine-tuned model loaded successfully


In [ ]:
!pip install pydub

In [ ]:
from IPython.display import Javascript, display, Audio
from google.colab import output
from base64 import b64decode
from io import BytesIO
from pydub import AudioSegment # You might need to install pydub: !pip install pydub

# JavaScript code to record audio
RECORD_AUDIO_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})

var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async () => {
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record_audio_colab(duration_seconds=5):
  """Records audio from the browser's microphone in Google Colab."""
  display(Javascript(RECORD_AUDIO_JS))
  print(f"Recording for {duration_seconds} seconds... Speak now!")
  s = output.eval_js(f'record({duration_seconds * 1000})') # Convert seconds to milliseconds
  print("Recording finished.")
  b = b64decode(s.split(',')[1])
  audio = AudioSegment.from_file(BytesIO(b))
  return audio

# --- How to use it ---

# 1. Install pydub if you haven't already
# !pip install pydub


## Test

Finally. Running this cell will record audio for 10seconds to test transcription

In [ ]:
print("starting recording for 10 seconds now...")
# 2. Record audio
recording = record_audio_colab(duration_seconds=10)

# 3. You can now work with the 'recording' AudioSegment object
# For example, to save it as a WAV file:
recording.export(WAVE_OUTPUT_FILENAME, format="wav")

# You can also play it back in Colab:
display(Audio(WAVE_OUTPUT_FILENAME))

# Or access its properties, e.g., duration
print(f"Recorded audio duration: {recording.duration_seconds} seconds")

starting recording for 10 seconds now...


<IPython.core.display.Javascript object>

Recording for 10 seconds... Speak now!
Recording finished.


Recorded audio duration: 9.42 seconds


In [ ]:
# Transcribe audio
print("Transcribing...")
audio_data, sr = librosa.load(WAVE_OUTPUT_FILENAME, sr=16000)
result = pipe(audio_data)
transcription = result['text'].strip()
print("Transcription:", transcription)

Transcribing...


/usr/local/lib/python3.11/dist-packages/transformers/models/whisper/generation_whisper.py:604: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257], 'f